In [2]:
!pip install xgboost joblib seaborn

Defaulting to user installation because normal site-packages is not writeable


In [3]:
!python -m pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [5]:
import sys
print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [6]:
import sys

!{sys.executable} -m pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable


In [7]:
import kagglehub

print("KaggleHub Installed Successfully!")

KaggleHub Installed Successfully!


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [9]:
transaction = pd.read_csv("datasets/train_transaction.csv")

identity = pd.read_csv("datasets/train_identity.csv")

print(transaction.shape)

print(identity.shape)

(590540, 394)
(144233, 41)


In [10]:
df = transaction.merge(
    identity,
    on="TransactionID",
    how="left"
)

print(df.shape)

df.head()

(590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [11]:
df["isFraud"].value_counts()

isFraud
0    569877
1     20663
Name: count, dtype: int64

In [12]:
df["isFraud"].value_counts(normalize=True)


isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

In [13]:
print(df.shape)

(590540, 434)


In [14]:
missing = (df.isnull().sum() / len(df)) * 100

missing = missing.sort_values(ascending=False)

missing.head(20)

id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_03    88.768923
id_04    88.768923
D6       87.606767
id_33    87.589494
id_10    87.312290
dtype: float64

In [15]:
threshold = 80

drop_cols = missing[missing > threshold].index

print("Columns to drop:", len(drop_cols))

df.drop(columns=drop_cols, inplace=True)

print("New Shape:", df.shape)

Columns to drop: 74
New Shape: (590540, 360)


In [16]:
numeric_cols = df.select_dtypes(include=['number']).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [17]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [18]:
df.isnull().sum().sum()

0

In [19]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object']).columns

encoders = {}

for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))
    encoders[col] = encoder

print("Categorical columns encoded:", len(categorical_cols))

Categorical columns encoded: 26


In [20]:
X = df.drop(columns=["TransactionID", "isFraud"])

y = df["isFraud"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (590540, 358)
Target Shape: (590540,)


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(472432, 358)
(118108, 358)


In [22]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Scale Pos Weight:", scale_pos_weight)

Scale Pos Weight: 27.580278281911674


In [23]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, random_state=42, ...)

In [24]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]


In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.9374555491583973
Precision: 0.33807722929936307
Recall   : 0.8219211226711831
F1 Score : 0.4790917424723221
ROC AUC  : 0.95521718605437


In [26]:
import joblib

joblib.dump(model, "xgboost_model.pkl")

print("XGBoost model saved successfully!")

XGBoost model saved successfully!


In [27]:
import joblib

# Save feature names
joblib.dump(X.columns.tolist(), "feature_names.pkl")

# Save label encoders
joblib.dump(encoders, "label_encoders.pkl")

# Save median values
joblib.dump(
    df.select_dtypes(include=["number"]).median(),
    "median_values.pkl"
)

print("All preprocessing files saved successfully!")

All preprocessing files saved successfully!
